# Azure Blob Storage Tutorial in Python


In [ ]:
# Note - you have to login via 'azure login'
# Install in case you have no downloaded requirements.txt

# !pip install azure-storage-blob azure-identity


In [ ]:
import os
import subprocess
import platform
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

In [6]:
# 1. Fetch from OS Environment Variable (Set this in your terminal/shell first)
# Command to set it: export AZURE_STORAGE_ACCOUNT_NAME="youraccountname"
STORAGE_ACCOUNT_NAME = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")

# 2. Safety Check: If not in OS, fall back or raise error
if not STORAGE_ACCOUNT_NAME:
    # Option A: Manual fallback for local debugging
    STORAGE_ACCOUNT_NAME = "my_dev_storage_account" 
    print(f"⚠️ Warning: AZURE_STORAGE_ACCOUNT_NAME not found in OS. Using fallback: {STORAGE_ACCOUNT_NAME}")
else:
    print(f"✅ Loaded Storage Account from OS: {STORAGE_ACCOUNT_NAME}")

try:
    # 3. Dynamic URL Fetching via Azure CLI
    # This ensures we get the correct endpoint regardless of the Azure Cloud (Public, Gov, etc.)
    cmd = ["az", "storage account", "show", "-n", STORAGE_ACCOUNT_NAME, "--query", "primaryEndpoints.blob", "-o", "tsv"]
    account_url = subprocess.check_output(cmd, text=True).strip()

    # 4. Auth & Client Init
    credential = DefaultAzureCredential()
    blob_service_client = BlobServiceClient(account_url=account_url, credential=credential)
    
    # 5. Connectivity Test
    properties = blob_service_client.get_service_properties()
    print(f"🚀 Successfully connected to: {account_url}")

except Exception as e:
    print(f"❌ Connection Failed: {e}")

⚠️ Warning: AZURE_STORAGE_ACCOUNT_NAME not found in OS. Using fallback: my_dev_storage_account
❌ Connection Failed: [Errno 2] No such file or directory: 'az'


In [ ]:
def create_container():
    container_client = blob_service_client.create_container(container_name)
    print('Container created.')

create_container()


In [ ]:
def upload_blob(local_file_path, blob_name):
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    with open(local_file_path, 'rb') as data:
        blob_client.upload_blob(data, overwrite=True)
    print('Uploaded.')

upload_blob('data/index.html', 'somefile.html')


In [ ]:
def list_blobs():
    container_client = blob_service_client.get_container_client(container=container_name)
    for blob in container_client.list_blobs():
        print(blob.name)

list_blobs()


In [ ]:
def download_blob(blob_name, download_file_path):
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    with open(download_file_path, 'wb') as file:
        file.write(blob_client.download_blob().readall())
    print('Downloaded.')

download_blob('somefile.html', 'data_download/index.html')
